In [5]:
import uproot
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import mplhep as hep


In [6]:
file = uproot.open("C:/Users/HP/Documents/Practica Profesional/Analisis BosonHiggs/Data Sets/Run2012B_DoubleMuParked.root")
tree = file['Events']

# Listar ramas (variables físicas)

print(tree.keys())
tree.keys()
print (['nMuon', 'Muon_pt', 'Muon_eta', 'Muon_phi', 'Muon_mass', 'Muon_charge','Muon_pfRelIso04_all', 'Muon_dxy', 'Muon_dz', 'Muon_dxyErr', 'Muon_dzErr'])

# Archivos de simulación Monte Carlo 
# Señal: H → ZZ* → 4μ
file_higgs = uproot.open("C:/Users/HP/Documents/Practica Profesional/Analisis BosonHiggs/Data Sets/SMHiggsToZZTo4L.root")
tree_higgs = file_higgs['Events']

# Fondo: ZZ → 4μ
file_zz = uproot.open("C:/Users/HP/Documents/Practica Profesional/Analisis BosonHiggs/Data Sets/ZZTo4mu.root")
tree_zz = file_zz['Events']

# Datos reales Run2012C 
file_c = uproot.open("C:/Users/HP/Documents/Practica Profesional/Analisis BosonHiggs/Data Sets/Run2012C_DoubleMuParked.root")
tree_c = file_c['Events']



['run', 'luminosityBlock', 'event', 'PV_npvs', 'PV_x', 'PV_y', 'PV_z', 'nMuon', 'Muon_pt', 'Muon_eta', 'Muon_phi', 'Muon_mass', 'Muon_charge', 'Muon_pfRelIso03_all', 'Muon_pfRelIso04_all', 'Muon_dxy', 'Muon_dxyErr', 'Muon_dz', 'Muon_dzErr', 'nElectron', 'Electron_pt', 'Electron_eta', 'Electron_phi', 'Electron_mass', 'Electron_charge', 'Electron_pfRelIso03_all', 'Electron_dxy', 'Electron_dxyErr', 'Electron_dz', 'Electron_dzErr', 'MET_pt', 'MET_phi']
['nMuon', 'Muon_pt', 'Muon_eta', 'Muon_phi', 'Muon_mass', 'Muon_charge', 'Muon_pfRelIso04_all', 'Muon_dxy', 'Muon_dz', 'Muon_dxyErr', 'Muon_dzErr']


In [7]:


def leer_arbol_a_df(tree, ramas):
    partes = []
    for batch in tree.iterate(ramas, step_size="100 MB", library="ak"):
        
        # Filtrar eventos con EXACTAMENTE 4 muones válidos con iso >= 0
        n_muones = ak.num(batch["Muon_pt"])
        
        # Solo eventos con al menos 4 muones
        mask = n_muones >= 4
        filtrado = batch[mask]
        
        # Tomar solo los primeros 4
        m1 = filtrado[:, 0]
        m2 = filtrado[:, 1]
        m3 = filtrado[:, 2]
        m4 = filtrado[:, 3]
        
        d = {
            "pt1": ak.to_numpy(m1["Muon_pt"]),   "eta1": ak.to_numpy(m1["Muon_eta"]),
            "phi1": ak.to_numpy(m1["Muon_phi"]),  "charge1": ak.to_numpy(m1["Muon_charge"]),
            "pt2": ak.to_numpy(m2["Muon_pt"]),   "eta2": ak.to_numpy(m2["Muon_eta"]),
            "phi2": ak.to_numpy(m2["Muon_phi"]),  "charge2": ak.to_numpy(m2["Muon_charge"]),
            "pt3": ak.to_numpy(m3["Muon_pt"]),   "eta3": ak.to_numpy(m3["Muon_eta"]),
            "phi3": ak.to_numpy(m3["Muon_phi"]),  "charge3": ak.to_numpy(m3["Muon_charge"]),
            "pt4": ak.to_numpy(m4["Muon_pt"]),   "eta4": ak.to_numpy(m4["Muon_eta"]),
            "phi4": ak.to_numpy(m4["Muon_phi"]),  "charge4": ak.to_numpy(m4["Muon_charge"]),
        }
        if "Muon_pfRelIso04_all" in ramas:
            d.update({
                "iso1": ak.to_numpy(m1["Muon_pfRelIso04_all"]),
                "dxy1": ak.to_numpy(m1["Muon_dxy"]), "dz1": ak.to_numpy(m1["Muon_dz"]),
                "dxyErr1": ak.to_numpy(m1["Muon_dxyErr"]), "dzErr1": ak.to_numpy(m1["Muon_dzErr"]),
                "iso2": ak.to_numpy(m2["Muon_pfRelIso04_all"]),
                "dxy2": ak.to_numpy(m2["Muon_dxy"]), "dz2": ak.to_numpy(m2["Muon_dz"]),
                "dxyErr2": ak.to_numpy(m2["Muon_dxyErr"]), "dzErr2": ak.to_numpy(m2["Muon_dzErr"]),
                "iso3": ak.to_numpy(m3["Muon_pfRelIso04_all"]),
                "dxy3": ak.to_numpy(m3["Muon_dxy"]), "dz3": ak.to_numpy(m3["Muon_dz"]),
                "dxyErr3": ak.to_numpy(m3["Muon_dxyErr"]), "dzErr3": ak.to_numpy(m3["Muon_dzErr"]),
                "iso4": ak.to_numpy(m4["Muon_pfRelIso04_all"]),
                "dxy4": ak.to_numpy(m4["Muon_dxy"]), "dz4": ak.to_numpy(m4["Muon_dz"]),
                "dxyErr4": ak.to_numpy(m4["Muon_dxyErr"]), "dzErr4": ak.to_numpy(m4["Muon_dzErr"]),
            })
        partes.append(pd.DataFrame(d))
    return pd.concat(partes, ignore_index=True)

In [ ]:
# Separar columnas para simulaciones MC 
def preparar_mc(df_raw):
    # CAMBIO 1: Usamos "pt1" para hacer la copia inicial porque "Muon_pt" ya no existe como lista
    df = df_raw[df_raw["pt1"].notna()].copy()
    
    # ------------------------------------------------------------------
    # CAMBIO 2: SE ELIMINÓ EL CICLO FOR CON LOS .apply(lambda)
    # ¿Por qué? Porque 'pt1', 'eta1', 'charge1', etc., ya fueron creados 
    # y separados de forma ultra rápida en tu celda anterior.
    # ------------------------------------------------------------------
    
    # CONSERVADO: Filtro de carga neta cero (2 pares de muones de carga opuesta)
    df = df[(df["charge1"]+df["charge2"]+df["charge3"]+df["charge4"]) == 0].copy()
    return df

# CONSERVADO: Llamadas a las funciones
df_higgs = preparar_mc(df_higgs_raw)
df_zz    = preparar_mc(df_zz_raw)

# CONSERVADO: Prints finales
print("Eventos Higgs MC:", len(df_higgs))
print("Eventos ZZ MC:   ", len(df_zz))

NameError: name 'df_higgs_raw' is not defined

: 

In [ ]:

df_4mu = df  

print("¡Listo! Datos reales estructurados en columnas individuales.")
print(f"Total de eventos listos para el análisis: {len(df_4mu):,}")

In [ ]:
# Valores únicos de las cargas (Para confirmar que hay muones + y -)

print("Valores únicos en charge1:", df_4mu["charge1"].unique())
print("Valores únicos en charge2:", df_4mu["charge2"].unique())
print("Valores únicos en charge3:", df_4mu["charge3"].unique())
print("Valores únicos en charge4:", df_4mu["charge4"].unique())

In [ ]:
# Filtrar pares de muones con cargas opuestas 

df_pairs = df_4mu[(df_4mu["charge1"] + df_4mu["charge2"] + df_4mu["charge3"] + df_4mu["charge4"]) == 0].copy()


In [ ]:
# Verificación de eventos totales con cuatro muones 

print("Eventos con cuatro muones:", len(df_4mu))
print("Eventos con cuatro muones y carga neta cero:", len(df_pairs))

In [ ]:
# Tabla de flujo de cortes

# Calcular sip3d ← NUEVO: parámetro de impacto 3D significativo (skim.cxx)
for i in range(1, 5):
    ip3d = np.sqrt(df_pairs[f"dxy{i}"]**2 + df_pairs[f"dz{i}"]**2)
    sig  = np.sqrt(df_pairs[f"dxyErr{i}"]**2 + df_pairs[f"dzErr{i}"]**2)
    df_pairs[f"sip3d{i}"] = ip3d / sig

mask_pt  = (df_pairs["pt1"] > 5) & (df_pairs["pt2"] > 5) & (df_pairs["pt3"] > 5) & (df_pairs["pt4"] > 5)
mask_eta = (df_pairs["eta1"].abs() < 2.4) & (df_pairs["eta2"].abs() < 2.4) & (df_pairs["eta3"].abs() < 2.4) & (df_pairs["eta4"].abs() < 2.4)
mask_iso = (df_pairs["iso1"]<0.40) & (df_pairs["iso2"]<0.40) & (df_pairs["iso3"]<0.40) & (df_pairs["iso4"]<0.40)  
mask_sip = (df_pairs["sip3d1"]<4) & (df_pairs["sip3d2"]<4) & (df_pairs["sip3d3"]<4) & (df_pairs["sip3d4"]<4)      
mask_dxy = (df_pairs["dxy1"].abs()<0.5) & (df_pairs["dxy2"].abs()<0.5) & (df_pairs["dxy3"].abs()<0.5) & (df_pairs["dxy4"].abs()<0.5)  
mask_dz  = (df_pairs["dz1"].abs()<1.0)  & (df_pairs["dz2"].abs()<1.0)  & (df_pairs["dz3"].abs()<1.0)  & (df_pairs["dz4"].abs()<1.0)  

df_final = df_pairs[mask_pt & mask_eta & mask_iso & mask_sip & mask_dxy & mask_dz].copy()  

n_total = len(df_pairs)
n_pt    = len(df_pairs[mask_pt])
n_eta   = len(df_pairs[mask_pt & mask_eta])
n_iso   = len(df_pairs[mask_pt & mask_eta & mask_iso])    
n_sip   = len(df_pairs[mask_pt & mask_eta & mask_iso & mask_sip])  
n_final = len(df_final) 

data_flujo = {
    'Corte aplicado': [
        'Sin filtros (4 muones)',
        'pT Líderes (>5, 5, 5, 5 GeV)',
        'Aceptancia Detector (|eta| < 2.4)',
        'Aislamiento iso < 0.40',          
        'Parámetro impacto sip3d < 4',     
        '|dxy| < 0.5 y |dz| < 1.0'        
    ],
    'Eventos Restantes': [n_total, n_pt, n_eta, n_iso, n_sip, n_final],  
    'Aceptancia (%)': [
        100.0,
        (n_pt    / n_total) * 100,
        (n_eta   / n_total) * 100,
        (n_iso   / n_total) * 100,   
        (n_sip   / n_total) * 100,   
        (n_final / n_total) * 100    
    ]
}
tabla_flujo = pd.DataFrame(data_flujo)
print("\nTabla de Flujo de Cortes (Análisis de Higgs 4l)")
print(tabla_flujo)




In [ ]:
# Estilo CMS
hep.style.use("CMS")

# Parámetros Físicos
m_mu = 0.10566  # Masa del muón [GeV/c²]

# ── Función masa invariante ────────────────────────────────────────
def calcular_masa(df):
    px = 0; py = 0; pz = 0; E = 0
    for i in range(1, 5):
        pxi = df[f"pt{i}"] * np.cos(df[f"phi{i}"])
        pyi = df[f"pt{i}"] * np.sin(df[f"phi{i}"])
        pzi = df[f"pt{i}"] * np.sinh(df[f"eta{i}"])
        Ei  = np.sqrt(pxi**2 + pyi**2 + pzi**2 + m_mu**2)
        px += pxi; py += pyi; pz += pzi; E += Ei
    return np.sqrt(E**2 - (px**2 + py**2 + pz**2))

# ── Calcular sip3d para datos reales ──────────────────────────────
for i in range(1, 5):
    ip3d = np.sqrt(df_pairs[f"dxy{i}"]**2 + df_pairs[f"dz{i}"]**2)
    sig  = np.sqrt(df_pairs[f"dxyErr{i}"]**2 + df_pairs[f"dzErr{i}"]**2)
    df_pairs[f"sip3d{i}"] = ip3d / sig

# ── Filtros datos reales ───────────────────────────────────────────
mask_pt     = True
mask_eta    = True
mask_iso    = True
mask_sip    = True
mask_dxy    = True
mask_dz     = True

for i in range(1, 5):
    iso_rel  = df_pairs[f"iso{i}"] / df_pairs[f"pt{i}"]
    mask_pt  &= (df_pairs[f"pt{i}"] > 5)
    mask_eta &= (df_pairs[f"eta{i}"].abs() < 2.4)
    mask_iso &= (df_pairs[f"iso{i}"] >= 0) & (iso_rel < 0.40)
    mask_sip &= (df_pairs[f"sip3d{i}"] < 4)
    mask_dxy &= (df_pairs[f"dxy{i}"].abs() < 0.5)
    mask_dz  &= (df_pairs[f"dz{i}"].abs() < 1.0)

mask_charge = (df_pairs["charge1"] + df_pairs["charge2"] +
               df_pairs["charge3"] + df_pairs["charge4"]) == 0

df_data_f = df_pairs[
    mask_pt & mask_eta & mask_iso & mask_sip & mask_dxy & mask_dz & mask_charge
].copy()

# ── Filtros MC ─────────────────────────────────────────────────────
mask_mc = lambda d: (
    (d["pt1"]>5) & (d["pt2"]>5) & (d["pt3"]>5) & (d["pt4"]>5) &
    (d["eta1"].abs()<2.4) & (d["eta2"].abs()<2.4) &
    (d["eta3"].abs()<2.4) & (d["eta4"].abs()<2.4) &
    ((d["charge1"]+d["charge2"]+d["charge3"]+d["charge4"]) == 0)
)

df_higgs_f = df_higgs[mask_mc(df_higgs)].copy()
df_zz_f    = df_zz[mask_mc(df_zz)].copy()

# ── Calcular masa ──────────────────────────────────────────────────
df_data_f["M"]  = calcular_masa(df_data_f)
df_higgs_f["M"] = calcular_masa(df_higgs_f)
df_zz_f["M"]    = calcular_masa(df_zz_f)

print("Datos reales filtrados:", len(df_data_f))
print("Higgs MC filtrado:     ", len(df_higgs_f))
print("ZZ MC filtrado:        ", len(df_zz_f))

# ── Pesos ──────────────────────────────────────────────────────────
lumi    = 11.58 * 1000
w_higgs = 0.0065 / 299973.0  * lumi
w_zz    = 0.077  / 1499064.0 * 1.386 * lumi

print(f"Peso Higgs: {w_higgs:.4f}")
print(f"Peso ZZ:    {w_zz:.4f}")

# ── Gráfico ────────────────────────────────────────────────────────
plt.figure(figsize=(10, 7))

plt.hist(df_zz_f["M"], bins=36, range=(70, 180),
         weights=np.full(len(df_zz_f), w_zz),
         label=r"$Z\gamma^*$, ZZ", color="#67C8F2", histtype='stepfilled')

plt.hist(df_higgs_f["M"], bins=36, range=(70, 180),
         weights=np.full(len(df_higgs_f), w_higgs),
         label=r"$m_H$ = 125 GeV", color="red", histtype='step', linewidth=2)

counts, bin_edges = np.histogram(df_data_f["M"], bins=36, range=(70, 180))
bin_centers = (bin_edges[1:] + bin_edges[:-1]) / 2
plt.errorbar(bin_centers, counts, yerr=np.sqrt(counts),
             fmt='o', color='black', markersize=5,
             capsize=3, elinewidth=1, linestyle='none', label='Data')

plt.title("CMS Open Data", loc='left', fontweight='bold', fontsize=15)
plt.title(r"11.6 fb$^{-1}$ (8 TeV)", loc='right', fontsize=12)
plt.xlabel(r"$m_{4\mu}$ [GeV]", fontsize=14)
plt.ylabel("Events / 3 GeV", fontsize=14)
plt.xlim(70, 180)
plt.ylim(0, counts.max() * 1.4)
plt.legend(loc='upper right', frameon=False)
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
print("df_pairs total:", len(df_pairs))
print("iso1 válido (>=0):", (df_pairs["iso1"] >= 0).sum())
print("iso2 válido (>=0):", (df_pairs["iso2"] >= 0).sum())
print("iso3 válido (>=0):", (df_pairs["iso3"] >= 0).sum())
print("iso4 válido (>=0):", (df_pairs["iso4"] >= 0).sum())
print("Los 4 iso válidos:", (
    (df_pairs["iso1"] >= 0) & (df_pairs["iso2"] >= 0) &
    (df_pairs["iso3"] >= 0) & (df_pairs["iso4"] >= 0)
).sum())

In [ ]:
# ¿Cuántos eventos pasan iso < 0.40 en los 4 muones?
mask_test = True
for i in range(1, 5):
    mask_test &= (df_pairs[f"iso{i}"] >= 0) & (df_pairs[f"iso{i}"] < 0.40)
print("Pasan iso<0.40 los 4 muones:", mask_test.sum())

# Ver iso vs pt para entender la escala
print("\nRelación iso1/pt1 (debería ser ~0.1 si es relativo):")
validos = df_pairs[df_pairs["iso1"] >= 0]
print((validos["iso1"] / validos["pt1"]).describe())

In [ ]:
# Calcular sip3d
for i in range(1, 5):
    ip3d = np.sqrt(df_pairs[f"dxy{i}"]**2 + df_pairs[f"dz{i}"]**2)
    sig  = np.sqrt(df_pairs[f"dxyErr{i}"]**2 + df_pairs[f"dzErr{i}"]**2)
    df_pairs[f"sip3d{i}"] = ip3d / sig

# Filtros
mask_iso = True
mask_pt  = True
mask_eta = True
mask_sip = True
mask_dxy = True
mask_dz  = True

for i in range(1, 5):
    iso_relativo = df_pairs[f"iso{i}"] / df_pairs[f"pt{i}"]   # ← conversión clave
    mask_iso &= (iso_relativo < 0.40) & (df_pairs[f"iso{i}"] >= 0)
    mask_pt  &= (df_pairs[f"pt{i}"] > 5)
    mask_eta &= (df_pairs[f"eta{i}"].abs() < 2.4)
    mask_sip &= (df_pairs[f"sip3d{i}"] < 4)
    mask_dxy &= (df_pairs[f"dxy{i}"].abs() < 0.5)
    mask_dz  &= (df_pairs[f"dz{i}"].abs() < 1.0)

mask_charge = (
    (df_pairs["charge1"] + df_pairs["charge2"] +
     df_pairs["charge3"] + df_pairs["charge4"]) == 0
)

df_data_f = df_pairs[mask_pt & mask_eta & mask_iso & mask_sip & mask_dxy & mask_dz & mask_charge].copy()

df_higgs_f = df_higgs[(df_higgs["pt1"]>5) & (df_higgs["pt2"]>5) &
                      (df_higgs["pt3"]>5) & (df_higgs["pt4"]>5) &
                      (df_higgs["eta1"].abs()<2.4) & (df_higgs["eta2"].abs()<2.4) &
                      (df_higgs["eta3"].abs()<2.4) & (df_higgs["eta4"].abs()<2.4)].copy()

df_zz_f = df_zz[(df_zz["pt1"]>5) & (df_zz["pt2"]>5) &
                (df_zz["pt3"]>5) & (df_zz["pt4"]>5) &
                (df_zz["eta1"].abs()<2.4) & (df_zz["eta2"].abs()<2.4) &
                (df_zz["eta3"].abs()<2.4) & (df_zz["eta4"].abs()<2.4)].copy()

df_data_f["M"]  = calcular_masa(df_data_f)
df_higgs_f["M"] = calcular_masa(df_higgs_f)
df_zz_f["M"]    = calcular_masa(df_zz_f)

print("Datos reales filtrados:", len(df_data_f))
print("Higgs MC filtrado:     ", len(df_higgs_f))
print("ZZ MC filtrado:        ", len(df_zz_f))
